# Lab 1 — TCP grezzo e MQTT

## Prima di iniziare

**Teoria di riferimento**

- Blocco 0.1, «Scope, transport and versions»: MQTT chiede un flusso di byte ordinato e senza perdite.
- Blocco 0.1, «Remaining Length encoding»: come MQTT marca la fine di un pacchetto.
- Blocco 0.1, «QoS 0, 1 and 2 exchanges»: i pacchetti di QoS 0, 1 e 2 sui due salti.
- Blocco 0.1, «Header and payload size», «Per-message framing overhead» e «Three payloads, the same framing».

**Che cosa vediamo oggi**

1. TCP consegna byte, non messaggi: il confine fra due messaggi lo scrive l'applicazione.
2. La stessa finestra di un secondo costa da 48 byte a 6 kB secondo come la scriviamo.
3. Quanti byte mette MQTT attorno a un messaggio, a ogni QoS, contro TCP grezzo, e chi paga un sottoscrittore in più.

**Lo strumento.** `Tap` sta fra un client e un server e copia i byte nei due versi,
registrandoli. Vede esattamente ciò che l'applicazione consegna a TCP; non vede le
intestazioni TCP, IP e di collegamento aggiunte sotto. Quando servono, il notebook le
**calcola** (20 byte di TCP e 40 di IPv6 per segmento, senza opzioni) e lo dice.

```
client ──► Tap (127.0.0.1:porta) ──► server: un Receiver, o il broker su localhost:1883
       ◄──                       ◄──
```

Il dataset del corso non serve: la finestra di dati è sintetica.

**Prima di eseguire**: un broker Mosquitto su `localhost:1883` (`mosquitto -v`, Lab 0).

**Tempo**: 2 ore. Prima di iniziare 15 min, Parte 1 20, Parte 2 25, Parte 3 30, Parte 4 20,
chiusura 10.

### Dove va questo laboratorio

Che cosa costa mandare una misura sulla rete, byte per byte?

| | domanda | output atteso |
|---|---|---|
| Parte 1 | TCP consegna messaggi o byte? | Byte: i confini fra i messaggi li scrive l'applicazione. |
| Parte 2 | Quanto pesa la stessa finestra in formati diversi? | Da 48 byte (un riassunto) a 6 kB (JSON): il formato è una scelta di progetto. |
| Parte 3 | Quanti byte aggiunge MQTT, a ogni QoS? | Pochi byte di intestazione, ma più pacchetti con QoS 1 e 2. |
| Parte 4 | Che cosa costa un secondo salto, o un sottoscrittore in più? | Ogni sottoscrittore riceve la sua copia: paga il broker, non il nodo. |

In [ ]:
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from iomtlab.config import REPO_ROOT
from iomtlab.net import (BROKER, TOPIC, IPV6_TCP_HEADERS, broker_available, synthetic_window,
                         pack_window, unpack_window, encode_json, publish_size,
                         recv_sizes, framed_transfer, encoding_sizes, mqtt_cost,
                         overhead_table, path_table, fanout_table,
                         plot_recv, plot_encodings, plot_overhead, plot_paths)
# le parentesi tonde permettono di andare a capo in una lista lunga di nomi da importare

WORK = REPO_ROOT / "work"  # la cartella dei risultati
WORK.mkdir(exist_ok=True)
# prima di tutto controllo che il broker risponda sulla 1883
# BROKER è una tupla ("localhost", 1883): *BROKER la «spacchetta» in due argomenti separati
assert broker_available(*BROKER), "nessun broker su localhost:1883: avvia Mosquitto (`mosquitto -v`)" # assert ferma la cella con questo messaggio se la condizione è falsa
print("broker raggiungibile su %s:%d; topic delle slide: %r, %d caratteri" % (*BROKER, TOPIC, len(TOPIC))) # formattazione con %: %s testo, %d intero, %r il valore fra virgolette

## Parte 1 — TCP consegna byte, non messaggi

> **Domanda.** TCP consegna messaggi o byte? **Output atteso.** Byte: i confini fra i messaggi li scrive l'applicazione.

**Che cosa facciamo.** Il mittente manda 10 messaggi da 11 byte ciascuno, con 10 `sendall` separate.
Il ricevitore aspetta 0,3 secondi prima di cominciare a leggere; poi chiama `recv` finché il mittente
non chiude e, ogni volta, si segna quanti byte ha ricevuto.

**Che cosa ci aspettiamo.** Se TCP conservasse i messaggi, vedremmo 10 `recv` da 11 byte. Invece in
0,3 secondi i 10 messaggi sono già tutti arrivati: una sola `recv` restituisce 110 byte, e dai byte non
si capisce più dove finiva un messaggio e cominciava il successivo.

**Perché ci interessa.** Chi scrive un protocollo deve dire lui dove finisce ogni messaggio. Nella
seconda cella mettiamo davanti a ogni messaggio 2 byte con la sua lunghezza: il ricevitore legge
prima la lunghezza, poi esattamente quel numero di byte. MQTT fa la stessa cosa con la Remaining
Length (Blocco 0.1, «Remaining Length encoding»; Parte 3).

In [ ]:
import socket
import threading
import time

ricevuti = []  # quanti byte restituisce ogni recv

def ricevitore(server):
    conn, _ = server.accept()     # aspetto che il mittente si colleghi: conn è il socket della connessione
    time.sleep(0.3)               # il ricevitore arriva in ritardo di 0,3 s
    while True:
        dati = conn.recv(65536)   # prendo quello che c'è, al massimo 65536 byte
        if not dati:              # b"": il mittente ha chiuso
            break
        ricevuti.append(len(dati))
    conn.close()

server = socket.create_server(("127.0.0.1", 0))   # un socket in ascolto; porta 0: la sceglie il sistema
porta = server.getsockname()[1]                   # la porta scelta
t = threading.Thread(target=ricevitore, args=(server,))  # il ricevitore gira su un altro thread
t.start()

mittente = socket.create_connection(("127.0.0.1", porta))  # il mittente si collega
for k in range(10):
    mittente.sendall(b"A" * 11)   # dieci messaggi da 11 byte
mittente.close()

t.join()                          # aspetto che il ricevitore finisca
server.close()
print(f"10 sendall da 11 byte  ->  {len(ricevuti)} recv: {ricevuti} byte")
plot_recv([11] * 10, ricevuti)
plt.show()

Ora gli stessi messaggi, ma di lunghezze diverse, con **2 byte di lunghezza davanti** a ognuno.
`struct.pack("!H", n)` scrive il numero `n` in 2 byte (big-endian); `struct.unpack` fa il contrario.
Il ricevitore arriva ancora in ritardo e trova tutto attaccato, ma leggendo prima la lunghezza
ritrova i messaggi uno per uno.

In [ ]:
import struct

def leggi_esattamente(conn, n):
    # recv può restituire meno byte di quelli chiesti: richiamo finché non ne ho n
    dati = b""
    while len(dati) < n:
        pezzo = conn.recv(n - len(dati))
        if not pezzo:              # b"": il mittente ha chiuso
            return None
        dati += pezzo
    return dati

messaggi_ricevuti = []

def ricevitore_con_lunghezza(server):
    conn, _ = server.accept()
    time.sleep(0.3)                          # anche qui il ricevitore arriva in ritardo
    while True:
        testa = leggi_esattamente(conn, 2)   # 1. leggo i 2 byte della lunghezza
        if testa is None:
            break
        n = struct.unpack("!H", testa)[0]    # dai 2 byte al numero n; unpack restituisce una tupla, [0] il primo valore
        messaggi_ricevuti.append(leggi_esattamente(conn, n))   # 2. leggo esattamente n byte: un messaggio intero
    conn.close()

msgs = [json.dumps({"seq": k, "nota": "x" * k}).encode() for k in range(10)] # list comprehension: 10 messaggi di lunghezze diverse
# json.dumps trasforma il dizionario in testo JSON; encode() trasforma il testo in byte, che è ciò che si manda sulla rete

server = socket.create_server(("127.0.0.1", 0))
porta = server.getsockname()[1]
t = threading.Thread(target=ricevitore_con_lunghezza, args=(server,))
t.start()

mittente = socket.create_connection(("127.0.0.1", porta))
for m in msgs:
    mittente.sendall(struct.pack("!H", len(m)) + m)   # davanti a ogni messaggio, 2 byte con la sua lunghezza
mittente.close()

t.join()
server.close()
print(f"{len(msgs)} messaggi mandati, {len(messaggi_ricevuti)} ricevuti, identici: {messaggi_ricevuti == msgs}")
print("il primo messaggio:", messaggi_ricevuti[0])
print("la sua lunghezza, scritta davanti:", struct.pack("!H", len(msgs[0])), "=", len(msgs[0]), "byte")

### Prova a cambiare

Si cambiano solo i valori fra le due righe `PARAMETRI`, poi si riesegue la cella.

**Prima di rieseguire: che cosa ti aspetti?** Con una pausa di 50 ms fra due invii e un ricevitore che legge subito, quante `recv` servono per dieci messaggi?

In [ ]:
# ── PARAMETRI ──────────────────────────────────────────────────────────────
N_MESSAGGI = 10             # quanti sendall faccio: da 2 a 50
BYTE_PER_MESSAGGIO = 11     # byte per sendall: da 1 a 60000
PAUSA_FRA_INVII_S = 0.05    # pausa del mittente fra due sendall: 0 oppure da 0.01 a 0.2 s
ATTESA_RICEVITORE_S = 0.0   # quanto aspetta il ricevitore prima della prima recv: da 0 a 1 s
SCRITTURE_SPEZZATE = True   # nel trasferimento con lunghezza, spezzo ogni frame in tre scritture
# ───────────────────────────────────────────────────────────────────────────
# recv_sizes e framed_transfer (libreria) fanno le stesse due cose di sopra, con questi tempi regolabili;
# framed_transfer passa anche dal Tap e può spezzare ogni messaggio in più scritture
inviati = [BYTE_PER_MESSAGGIO] * N_MESSAGGI
ricevuti = recv_sizes(N_MESSAGGI, BYTE_PER_MESSAGGIO, PAUSA_FRA_INVII_S, ATTESA_RICEVITORE_S)
plot_recv(inviati, ricevuti)
plt.show()
print(f"{N_MESSAGGI} sendall  ->  {len(ricevuti)} recv")
msgs = [bytes(BYTE_PER_MESSAGGIO)] * N_MESSAGGI # bytes(n): n byte a zero
r = framed_transfer(msgs, split_writes=SCRITTURE_SPEZZATE)
print(f"con la lunghezza davanti: {r['received']} messaggi, identici: {r['identical']}, "
      f"{r['wire_bytes'] - r['payload_bytes']} byte in più del payload")

**Che cosa guardare.** Con la pausa, le `recv` sembrano rispettare i messaggi: è una coincidenza di tempi, non una proprietà di TCP, e sparisce appena il ricevitore è più lento del mittente. Con la lunghezza davanti i messaggi tornano identici in tutti i casi, anche con le scritture spezzate, e il costo è di 2 byte per messaggio. Con messaggi oltre 65535 byte la cella si ferma: 2 byte di lunghezza non bastano.

**Da discutere**

1. Su una rete reale, con perdite e ritrasmissioni, ti aspetti che il numero di `recv` per messaggio sia più o meno prevedibile che su `localhost`?
2. MQTT scrive la lunghezza con la Remaining Length, da 1 a 4 byte, invece che con 2 byte fissi: che cosa guadagna e che cosa paga (Blocco 0.1, «Remaining Length encoding»)?

## Parte 2 — Il formato del payload

> **Domanda.** Quanto pesa la stessa finestra in formati diversi? **Output atteso.** Da 48 byte (un riassunto) a 6 kB (JSON): il formato è una scelta di progetto.

Né TCP né MQTT dicono che cosa c'è nel payload (Blocco 0.1, «Three payloads, the same
framing»). Codifico la stessa finestra di un secondo, 50 campioni per 6 canali,
in cinque modi e conto i byte. La finestra è sintetica, con le ampiezze di un telefono in
tasca durante la camminata: oggi si misurano byte, non movimento. Il formato `int16` è quello
di `pack_window`: 11 byte di intestazione e ogni canale quantizzato sul fondo scala del
dispositivo. Il riassunto sono media e deviazione standard di ogni canale: 12 numeri.

In [ ]:
t0_ms, X = synthetic_window()                          # 50 campioni x 6 canali, deterministica; t0_ms è l'istante iniziale in millisecondi
sizes = encoding_sizes(X, t0_ms, decimals=3)           # dizionario: byte per finestra in ciascuna codifica
riassunto = np.concatenate([X.mean(axis=0), X.std(axis=0)]).astype(">f4").tobytes()
# axis=0: media e deviazione standard lungo i campioni, una per canale; ">f4" è float32 big-endian; tobytes() dà i byte grezzi
sizes.pop("features")                                  # tolgo la codifica che usa le feature (le vedremo nel Lab 2)
sizes["riassunto, 12 float32"] = len(riassunto)        # e aggiungo il riassunto: 12 numeri x 4 byte
plot_encodings(sizes, windows_per_s=2)                 # passo di 25 campioni a 50 Hz: 2 finestre al secondo
plt.show()
_, _, Xd = unpack_window(pack_window(0, t0_ms, X))     # codifico in int16 e decodifico; _ è il nome per un valore che non ci serve
print("errore massimo di int16 per canale:", np.abs(Xd - X).max(axis=0).round(5), "(m/s², rad/s)")
print("inizio del JSON:", encode_json(0, t0_ms, X)[:70]) # [:70]: i primi 70 caratteri

### Scrivi tu — il payload di una decisione (circa 15 minuti)

**Teoria**: Blocco 0.1, «Per-message framing overhead». Invece dei dati, il nodo può mandare solo la sua conclusione, la *decisione*: un numero di classe (per esempio 2 = «sta camminando») con la sua confidenza. Il contenuto è pochi byte, e il formato decide quanti.

**Che cosa scrivere.** `pack_decision(t_ms, classe, confidenza)` restituisce **6 byte**,
big-endian, nell'ordine:

| campo | tipo `struct` | byte | valore |
|---|---|---|---|
| istante della decisione in ms | `I` (uint32) | 4 | `t_ms` modulo 2³² |
| classe | `B` (uint8) | 1 | da 0 a 4 |
| confidenza | `B` (uint8) | 1 | in percento intero, da 0 a 100 |

`unpack_decision(b)` fa il contrario e restituisce `(t_ms, classe, confidenza)` con la
confidenza fra 0 e 1. Il formato è `"!IBB"`: `!` vuol dire big-endian e niente riempimento.

Si scrive solo dentro la funzione; la cella successiva la controlla. Se il tempo finisce, la cella di controllo usa la versione della libreria e il notebook prosegue.

In [ ]:
import struct

def pack_decision(t_ms, classe, confidenza):
    """Una decisione in 6 byte: t_ms uint32, classe uint8, confidenza in % uint8, big-endian."""
    # scrivi qui: struct.pack con il formato "!IBB"
    raise NotImplementedError

def unpack_decision(b):
    """Da 6 byte a (t_ms, classe, confidenza fra 0 e 1)."""
    raise NotImplementedError

In [ ]:
from iomtlab.net import pack_decision as rif_pack, unpack_decision as rif_unpack # le versioni della libreria, con un altro nome per non coprire le vostre
try:
    prove = [(0, 0, 0.0), (123456, 2, 0.934), (2**32 + 5, 4, 1.0), (987654321, 3, 0.5)] # lista di tuple (t_ms, classe, confidenza)
    for t, c, p in prove: # ogni tupla viene spacchettata in tre variabili
        assert pack_decision(t, c, p) == rif_pack(t, c, p), f"byte diversi per {(t, c, p)}"
        assert unpack_decision(rif_pack(t, c, p)) == rif_unpack(rif_pack(t, c, p))
    print("giusto: stessi byte della libreria su", len(prove), "decisioni")
except NotImplementedError: # la funzione non è ancora scritta
    pack_decision, unpack_decision = rif_pack, rif_unpack
    print("funzione non ancora scritta: uso quella della libreria (iomtlab.net.payload)")

In [ ]:
d = pack_decision(t0_ms, 2, 0.93)
js = json.dumps({"t": t0_ms, "c": 2, "p": 93}, separators=(",", ":")).encode() # separators senza spazi: il JSON più corto possibile
finestra = pack_window(0, t0_ms, X)
righe = []
for nome, pl in (("decisione, struct", d), ("decisione, JSON", js), ("finestra int16", finestra)):
    pub = publish_size(len(TOPIC), len(pl), qos=1) + 4          # PUBLISH a QoS 1 più il suo PUBACK (4 byte)
    filo = pub + 2 * IPV6_TCP_HEADERS                          # due segmenti TCP/IPv6, calcolati
    righe.append({"payload": nome, "byte di payload": len(pl), "byte MQTT": pub,
                  "byte sul filo (calcolati)": filo, "kbit/s a 2 decisioni/s": 2 * 8 * filo / 1000}) # una riga della tabella, come dizionario
display(pd.DataFrame(righe).set_index("payload")) # display mostra la tabella anche se non è l'ultima riga della cella

**Che cosa guardare.** La decisione in `struct` occupa 6 byte contro i 600 e oltre della finestra, ma sul filo la differenza si riduce: topic, intestazione MQTT e segmenti TCP/IPv6 sono gli stessi per ogni messaggio. È il conto di «Per-message framing overhead»: con un payload piccolo, l'intestazione è quasi tutto il messaggio.

**Da discutere**

1. Quanta informazione perdi arrotondando la confidenza al percento intero? Per che cosa la userebbe chi riceve?
2. La classe occupa un byte, ma 5 classi stanno in 3 bit: vale la pena comprimere il byte, visto il resto del messaggio?

## Parte 3 — MQTT sul filo, contro TCP grezzo

> **Domanda.** Quanti byte aggiunge MQTT, a ogni QoS? **Output atteso.** Pochi byte di intestazione, ma più pacchetti con QoS 1 e 2.

Mando payload di dimensioni diverse con la lunghezza davanti su TCP grezzo (Parte 1) e con
MQTT a QoS 0, 1 e 2, e misuro con il tap i byte per messaggio sul collegamento del nodo, nei
due versi (per MQTT solo i pacchetti del messaggio: PUBLISH e le conferme). Il PUBLISH misurato
si confronta con `publish_size`, calcolato dalla specifica (Blocco 0.1, «PUBLISH packet»; «Header and payload size»; Blocco 0.1, «Per-message framing overhead»).

In [ ]:
t0_ms, X = synthetic_window()
riassunto = np.concatenate([X.mean(axis=0), X.std(axis=0)]).astype(">f4").tobytes() # lo stesso riassunto della Parte 2
payloads = { # un dizionario: nome del payload -> i suoi byte
    "10 B di prova": b"0123456789", # b"...": una stringa di byte
    "riassunto, 12 float32": riassunto,
    "finestra int16": pack_window(0, t0_ms, X),
    "finestra JSON, 3 decimali": encode_json(0, t0_ms, X, decimals=3),
}
tab = overhead_table(payloads, TOPIC)                  # funzione della libreria: manda ogni payload e misura con il Tap i byte per messaggio
tab.to_csv(WORK / "lab01_overhead.csv")                # salvo la tabella anche su file, in work/
plot_overhead(tab)
plt.show()

# un PUBLISH a QoS 1 della finestra int16, byte per byte
win = payloads["finestra int16"]
print("PUBLISH QoS 1 misurato:", mqtt_cost(win, qos=1),
      "  previsto:", publish_size(len(TOPIC), len(win), 1),
      "= 1 + 2 + (2 +", len(TOPIC), ") + 2 +", len(win))
tab

### Prova a cambiare

Si cambiano solo i valori fra le due righe `PARAMETRI`, poi si riesegue la cella.

**Prima di rieseguire: che cosa ti aspetti?** Con il topic di 18 caratteri, fra 107 e 108 byte di payload il PUBLISH a QoS 0 cresce di un byte o di due?

In [ ]:
# ── PARAMETRI ──────────────────────────────────────────────────────────────
TOPIC_PROVA = TOPIC                     # il topic: prova "t" (1 carattere) o un topic di 60 caratteri
DIMENSIONI_PAYLOAD = [10, 100, 107, 108, 500]   # byte di payload da provare, da 0 a 60000
QOS = (0, 1, 2)                         # i livelli di QoS da misurare
# ───────────────────────────────────────────────────────────────────────────
prova = overhead_table({f"{n} B": bytes(n) for n in DIMENSIONI_PAYLOAD}, TOPIC_PROVA, QOS) # dict comprehension: un payload di n byte a zero per ogni dimensione
plot_overhead(prova)
plt.show()
prova.assign(**{f"extra_qos{q}": prova[f"qos{q}"] - prova["payload_B"] for q in QOS}) # assign aggiunge colonne; ** passa il dizionario come argomenti con nome

**Che cosa guardare.** Le colonne `extra_qos` sono costanti a ogni QoS, salvo il salto di un byte quando la Remaining Length passa da 1 a 2 byte (oltre 127 byte di resto del pacchetto). Il topic si paga per intero a ogni messaggio: un carattere, un byte. `publish_ok` vero vuol dire che il PUBLISH misurato è quello calcolato dalla specifica. TCP grezzo con la lunghezza costa 2 byte, ma non dà né topic, né conferme, né un broker.

**Da discutere**

1. Con un payload di 10 byte, quale parte del messaggio a QoS 1 è la più grande, e che cosa la accorcerebbe (Blocco 0.1, «Header and payload size»)?
2. TCP grezzo costa 2 byte per messaggio: che cosa dovrebbe scrivere l'applicazione per avere ciò che MQTT aggiunge?

## Parte 4 — Due salti e più sottoscrittori

> **Domanda.** Che cosa costa un secondo salto, o un sottoscrittore in più? **Output atteso.** Ogni sottoscrittore riceve la sua copia: paga il broker, non il nodo.

Un messaggio attraversa due salti: nodo → broker e broker → sottoscrittore, ognuno con il suo
scambio alla sua QoS (Blocco 0.1, «QoS 0, 1 and 2 exchanges»). Misuro con due tap, uno sul
collegamento del nodo e uno sul lato dei sottoscrittori: prima un sottoscrittore e 10 byte di
payload alle tre QoS, poi la finestra int16 a QoS 1 con 1, 2 e 4 sottoscrittori.

In [ ]:
path = path_table(b"0123456789")                       # 10 byte, un sottoscrittore, QoS 0, 1, 2
fan = fanout_table((1, 2, 4), qos=1, payload=win)      # finestra int16, QoS 1
plot_paths(path, fan)                                  # a sinistra i due salti, a destra il fan-out
plt.show()
display(path)
fan

### Prova a cambiare

Si cambiano solo i valori fra le due righe `PARAMETRI`, poi si riesegue la cella.

**Prima di rieseguire: che cosa ti aspetti?** Con 8 sottoscrittori, quanti byte per messaggio passano sul collegamento del nodo? E dal broker verso i sottoscrittori?

In [ ]:
# ── PARAMETRI ──────────────────────────────────────────────────────────────
SOTTOSCRITTORI = (1, 4, 8)  # quanti sottoscrittori provare: da 1 a 20
QOS_FANOUT = 2              # la QoS di pubblicazione e di sottoscrizione: 0, 1 o 2
PAYLOAD_FANOUT = win        # il payload: win (611 B) o payloads["riassunto, 12 float32"] (48 B)
# ───────────────────────────────────────────────────────────────────────────
fan2 = fanout_table(SOTTOSCRITTORI, qos=QOS_FANOUT, payload=PAYLOAD_FANOUT)
plot_paths(path, fan2)
plt.show()
fan2.assign(per_sottoscrittore=fan2["sottoscrittori"] / fan2.index) # byte dal broker divisi per il numero di sottoscrittori (l'indice della tabella)

**Che cosa guardare.** Il collegamento del nodo non cambia con il numero di sottoscrittori: il nodo pubblica una volta sola. Il lato del broker cresce linearmente, un messaggio completo per sottoscrittore. In una WSN il collegamento vincolato è quello del nodo; con TCP grezzo il nodo dovrebbe aprire una connessione e mandare una copia per ogni consumatore.

**Da discutere**

1. Se il collegamento vincolato fosse quello dei sottoscrittori e non quello del nodo, il fan-out nel broker sarebbe ancora un vantaggio?
2. A QoS 2 ogni salto ha il suo scambio di quattro pacchetti: il nodo sa quando il messaggio è arrivato al sottoscrittore?

## Prova a scrivere

**Il numero di oggi**: il flusso sul collegamento del nodo, in bit/s, per mandare ogni finestra in `int16`
a QoS 1, due finestre al secondo. Due versioni: con i byte applicativi misurati dal Tap, e con in più le
intestazioni TCP/IPv6 calcolate (un segmento per il PUBLISH, uno per il PUBACK, `IPV6_TCP_HEADERS` byte ciascuno).

`mqtt_cost(win, qos=1)` restituisce un dizionario con i byte di ogni pacchetto: la somma dei valori sono i byte per finestra.

In [ ]:
per_msg = mqtt_cost(win, qos=1)                        # PUBLISH e PUBACK, byte per finestra
# 1. b: la somma dei valori del dizionario per_msg (sum e .values())
# 2. bit_s: b byte, 8 bit per byte, 2 finestre al secondo
# 3. bit_s_ip: come bit_s, ma con 2 * IPV6_TCP_HEADERS byte in più per finestra
b = None
bit_s = None
bit_s_ip = None

grezzo = 50 * 6 * 16                                   # il flusso grezzo: 50 campioni al secondo, 6 canali, 16 bit
if b is None:                                          # finché non avete scritto i tre valori, la cella non dà errore
    print("da completare: scrivi b, bit_s e bit_s_ip")
else:
    print(f"{b} byte per finestra -> {bit_s} bit/s misurati, {bit_s_ip} bit/s con TCP/IPv6")
    print(f"rispetto al flusso grezzo di {grezzo} bit/s: {bit_s / grezzo:.2f} e {bit_s_ip / grezzo:.2f} volte")

Il nodo spende più del doppio del flusso grezzo a 50 Hz: con finestre sovrapposte al 50 %
ogni campione viaggia due volte, e a ogni finestra si aggiungono l'intestazione del formato,
il PUBLISH con il suo topic, il PUBACK e, sotto, i due segmenti TCP/IPv6.

**Da riportare**: il numero di oggi in bit/s, e una frase: che cosa cambieresti per portarlo
sotto il flusso grezzo.